In [32]:
import logging
logging.basicConfig(
    # make sure debug level logs are shown
    level=logging.DEBUG,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler()])


### Save paper PDF

Ensure that you can save the paper in a specific directory.

In [2]:
import arxiv_search
import os

download_dir = "test/pdfs/arxiv"
os.makedirs(download_dir, exist_ok=True)

# download attention is all you need paper: https://arxiv.org/pdf/2510.26641
arxiv_search.download_pdf("2510.26641",save_dir=download_dir)


2025-10-31 21:43:01,419 [DEBUG] Save dir:test/pdfs/arxiv
2025-10-31 21:43:01,421 [DEBUG] arXiv url: https://arxiv.org/pdf/2510.26641
2025-10-31 21:43:01,421 [DEBUG] Downloading https://arxiv.org/pdf/2510.26641 to test/pdfs/arxiv/2510.26641.pdf
2025-10-31 21:43:02,973 [DEBUG] Starting new HTTPS connection (1): arxiv.org:443
2025-10-31 21:43:03,027 [DEBUG] https://arxiv.org:443 "GET /pdf/2510.26641 HTTP/1.1" 200 1853
2025-10-31 21:43:03,030 [INFO] Saved: test/pdfs/arxiv/2510.26641.pdf


### Build a search URL

Lets use the custom function to build a search URL

In [34]:
topics = [
    "attention is all you need",
    "machine learning in the modern era"
]

urls = []

for topic in topics:
    urls.append(arxiv_search.build_arxiv_search_url(query=topic, size=100))

print(urls)

['https://arxiv.org/search/cs?query=attention+is+all+you+need&searchtype=all&abstracts=show&order=-announced_date_first&size=100', 'https://arxiv.org/search/cs?query=machine+learning+in+the+modern+era&searchtype=all&abstracts=show&order=-announced_date_first&size=100']


### Scrap 

Scrap data from the urls in the background and save to json file

In [35]:
import time
# Prepare directory
scraped_directory = "test/scraped"
os.makedirs(scraped_directory, exist_ok=True)

# output file
output_file = []

# run the scraper for each url
for index, url in enumerate(urls):
    file_location = f"{scraped_directory}/{index}_scraped.json"
    arxiv_search.run_scraper_in_background(url=url, output_file=file_location)
    time.sleep(3)

    output_file.append(file_location)

print(output_file)

2025-10-31 21:30:50,491 [DEBUG] Starting new HTTPS connection (1): arxiv.org:443
2025-10-31 21:30:50,491 [DEBUG] Scraping started in background. Results will be saved to test/scraped/0_scraped.json.
2025-10-31 21:30:50,531 [DEBUG] https://arxiv.org:443 "GET /search/cs?query=attention+is+all+you+need&searchtype=all&abstracts=show&order=-announced_date_first&size=100 HTTP/1.1" 200 1853
2025-10-31 21:30:50,535 [DEBUG] Saved 0 papers to test/scraped/0_scraped.json
2025-10-31 21:30:50,536 [DEBUG] Type of output file:<class 'str'>
2025-10-31 21:30:53,494 [DEBUG] Starting new HTTPS connection (1): arxiv.org:443
2025-10-31 21:30:53,495 [DEBUG] Scraping started in background. Results will be saved to test/scraped/1_scraped.json.
2025-10-31 21:30:53,616 [DEBUG] https://arxiv.org:443 "GET /search/cs?query=machine+learning+in+the+modern+era&searchtype=all&abstracts=show&order=-announced_date_first&size=100 HTTP/1.1" 200 1853
2025-10-31 21:30:53,621 [DEBUG] Saved 0 papers to test/scraped/1_scraped.

['test/scraped/0_scraped.json', 'test/scraped/1_scraped.json']


Scrape metadata from the json file

In [36]:
print(output_file)

['test/scraped/0_scraped.json', 'test/scraped/1_scraped.json']


In [37]:
import logging

pages_metadata = []

for index, value in enumerate[str](output_file):
    logging.debug(f"{index}.{value}")
    # extend the list (we only need one list)
    pages_metadata.append(arxiv_search.scrape_arxiv_details_from_json_threaded(value))

print(len(pages_metadata))

2025-10-31 21:30:56,518 [DEBUG] 0.test/scraped/0_scraped.json
2025-10-31 21:30:56,520 [DEBUG] Scraped 0 detailed entries using threads.
2025-10-31 21:30:56,520 [DEBUG] 1.test/scraped/1_scraped.json
2025-10-31 21:30:56,522 [DEBUG] Scraped 0 detailed entries using threads.


2


In [38]:
pages_metadata

[[], []]

In [39]:
# save the combined list to a file
# enriched data
enriched_data_dir = "test/enriched"
os.makedirs(enriched_data_dir, exist_ok=True)

metadata_file = f"{enriched_data_dir}/papers_metadata.json"
arxiv_search.save_arxiv_scraped_details(results=pages_metadata, output_file=metadata_file)

2025-10-31 21:30:56,543 [DEBUG] Saved cleaned results to test/enriched/papers_metadata.json
